# 03 — Audio Quality Control

Reproduit la section **Technical Validation** du papier (Nsumba et al., 2026) :

| Check du papier | Ce qu'on fait |
|---|---|
| Decodability & integrity | Decode each audio file, check sample rate / duration / channels |
| Silence & dropouts | RMS per frame, flag if the median falls below a threshold (failing mic) |
| Spectral sanity | Energy per band: 0-250 Hz / 250 Hz-2 kHz / 2-8 kHz, outliers > 3 MAD |
| Duplication | Hash MD5 sur le contenu audio |

We work on a sample (audio is heavy) - the logic is identical on the full dataset.

In [ ]:
from datasets import load_dataset, Audio
import numpy as np
import pandas as pd
import librosa
import hashlib
import io
import matplotlib.pyplot as plt

# HF token loaded from .env (gitignored)
from dotenv import load_dotenv
import os
load_dotenv('../.env')
HF_TOKEN = os.environ['HF_TOKEN']

# Streaming so as not to download 3 GB - we take N_AUDIO samples.
# decode=False: we take the raw bytes and decode them ourselves with librosa
# (avoids the torchcodec/PyTorch dependency required by recent versions of datasets)
N_AUDIO = 200
ds = load_dataset('Sunbird/urban-noise-uganda-61k', 'small', split='train',
                  streaming=True, token=HF_TOKEN)
ds = ds.cast_column('audio', Audio(decode=False))

In [ ]:
# ---- Check 1+2+3+4 en une passe ----
RMS_SILENCE_THRESHOLD = 1e-4   # median RMS below this = suspect (dead mic)

def band_energy(y, sr):
    """Energy in the paper's 3 bands: 0-250 Hz, 250-2k, 2k-8k."""
    S = np.abs(np.fft.rfft(y)) ** 2
    freqs = np.fft.rfftfreq(len(y), 1 / sr)
    total = S.sum() + 1e-12
    return (
        S[freqs < 250].sum() / total,
        S[(freqs >= 250) & (freqs < 2000)].sum() / total,
        S[(freqs >= 2000) & (freqs < 8000)].sum() / total,
    )

rows = []
for i, ex in enumerate(ds):
    if i >= N_AUDIO:
        break
    try:
        # Manual decoding of the .ogg bytes with librosa (16 kHz mono, as in the paper)
        audio_bytes = ex['audio']['bytes']
        y, sr = librosa.load(io.BytesIO(audio_bytes), sr=16000, mono=True)
        rms_frames = librosa.feature.rms(y=y)[0]
        low, mid, high = band_energy(y, sr)
        rows.append({
            'idx': i,
            'decodable': True,
            'duration_s': len(y) / sr,
            'sr': sr,
            'median_rms': float(np.median(rms_frames)),
            'band_low': low, 'band_mid': mid, 'band_high': high,
            'md5': hashlib.md5(audio_bytes).hexdigest(),
            'noise_dB': ex.get('noise_measurement'),
            'class': ex.get('class'),
        })
    except Exception as e:
        rows.append({'idx': i, 'decodable': False, 'error': str(e)})

qc = pd.DataFrame(rows)
print(f"{len(qc)} files analysed - {qc['decodable'].sum()} decodable")

In [ ]:
# ---- Quality flags (same criteria as the paper) ----
ok = qc[qc['decodable']].copy()

# Silence / micro mort
ok['flag_silence'] = ok['median_rms'] < RMS_SILENCE_THRESHOLD

# Spectral outliers: > 3 MAD from the median (the paper's exact criterion)
for band in ['band_low', 'band_mid', 'band_high']:
    med = ok[band].median()
    mad = (ok[band] - med).abs().median()
    ok[f'flag_{band}'] = (ok[band] - med).abs() > 3 * mad

# Doublons exacts par hash
ok['flag_duplicate'] = ok.duplicated(subset='md5', keep='first')

flags = [c for c in ok.columns if c.startswith('flag_')]
ok['any_flag'] = ok[flags].any(axis=1)

print('QC summary:')
for f in flags:
    print(f'  {f:18s}: {ok[f].sum():4d} files')
print(f'  {"TOTAL flagged":18s}: {ok["any_flag"].sum()} / {len(ok)}')

In [ ]:
# Visualisation: RMS distribution and energy per band
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(np.log10(ok['median_rms'] + 1e-12), bins=30, color='steelblue')
axes[0].axvline(np.log10(RMS_SILENCE_THRESHOLD), color='red', linestyle='--', label='Seuil silence')
axes[0].set_title('log10(median RMS)')
axes[0].legend()

ok[['band_low', 'band_mid', 'band_high']].boxplot(ax=axes[1])
axes[1].set_title('Energy per band (fraction)')

axes[2].hist(ok['duration_s'], bins=30, color='steelblue')
axes[2].set_title('Duration (s) - paper: >= 10 s')

plt.tight_layout()
plt.savefig('../results/figures/sunbird/audio_qc.png', dpi=150)
plt.show()

# Sauvegarde la liste propre (sans flags) pour la suite
ok[~ok['any_flag']].to_csv('../data/processed/uganda/audio_qc_passed.csv', index=False)
print(f"{(~ok['any_flag']).sum()} valid files saved")

## Note for the Hanoi measurements

The same QC cell will run on the field recordings - it is enough to replace
the source with the field recordings in `data/raw/kobo/`.